<a href="https://colab.research.google.com/github/a00377571/lsm-reconocimiento-mediapipe/blob/main/notebooks/Avance4.47.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Maestría en Inteligencia Artificial Aplicada**

**Curso: TC5035.10 - Proyecto Integrador**

Tecnológico de Monterrey

Profesor: Dr. Raúl Valente Ramírez Velarde

**Avance 4. Modelos alternativos**

---

*   **Estudiante:** Marco Antonio Arellano Hernández - A00377571
*   **Estudiante:** Miguel Octavio Ramírez Romero - A00828311
*   **Estudiante:** Hugo Enrique Navarro Villeda - A01796287

# Avance 4. Modelos alternativos

## Qué hace esta libreta

1. Extrae con MediaPipe **una fila de características por video** del dataset ITESO-MX-100 (forma de ambas manos, trayectoria de las muñecas respecto a los hombros y detección).
2. Compara **6 modelos individuales (no ensambles)** con los mismos folds de validación cruzada y reporta F1 macro, accuracy, recall macro y tiempos.
3. **Ajusta los dos mejores** con búsqueda en malla.
4. Elige el **modelo individual final** y lo evalúa una sola vez en un conjunto de prueba reservado.

## Decisiones de diseño

* **Unidad de observación: el video completo.** Es la misma decisión del Avance 2: una seña dinámica no se puede describir con un frame aislado, y así se evita que frames casi idénticos de un mismo video caigan en entrenamiento y prueba (fuga de información que inflaba el resultado del Avance 3).
* **Métrica principal: F1 macro.** Da el mismo peso a cada seña aunque haya clases con menos videos. Se reportan también accuracy y recall macro.
* **Preprocesamiento dentro de un `Pipeline`.** Imputación, filtro de varianza y escalado se ajustan solo con los datos de entrenamiento de cada fold, para evitar fuga de información.
* **Manos no detectadas = ausentes, no imputadas con media.** Es la conclusión del EDA del Avance 1. Se rellenan con 0 y el modelo conserva la proporción de frames detectados como característica.
* **Limitación:** los nombres de archivo del dataset (`1.mp4`, `2.mp4`...) no indican qué participante grabó cada video, por lo que la separación es estratificada por video y no por persona. Si el dataset trae metadatos de participante, conviene repetir la separación agrupando por persona.

## 1. Preparación del entorno

Se usa la misma versión de MediaPipe que en el Avance 2. Si Colab pide reiniciar la sesión después de instalar, se reinicia una vez y se ejecuta la libreta desde el inicio.

In [ ]:
%pip -q install "mediapipe==0.10.35" tqdm

In [ ]:
from pathlib import Path
import json
import random
import time
import urllib.request
import warnings

import cv2
import joblib
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from tqdm.auto import tqdm

from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             classification_report, f1_score, recall_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_validate, train_test_split)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style='whitegrid')
SEED = 42
print('MediaPipe:', mp.__version__)

## 2. Configuración

El dataset (unos 10 GB) se sube a Google Drive y se monta en Colab. Los resultados de la extracción se guardan en Drive después de cada lote de videos, de modo que si Colab se desconecta el proceso **continúa donde se quedó**.

* `N_SIGNS`: número de señas a usar. Con `10` el flujo completo corre rápido; con `None` se usan todas las carpetas disponibles.
* `MAX_VIDEOS_PER_SIGN`: limita videos por seña para pruebas rápidas (`None` = todos).
* `SKIP`: se procesa 1 de cada `SKIP` frames para acelerar la extracción.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ajusta esta ruta a donde subiste la carpeta ITESO-MX-100 en tu Drive.
DATASET_DIR = Path('/content/drive/MyDrive/ITESO-MX-100')
WORK_DIR = Path('/content/drive/MyDrive/lsm_avance4')
WORK_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_CSV = WORK_DIR / 'caracteristicas_videos.csv'

N_SIGNS = 10
MAX_VIDEOS_PER_SIGN = None
SKIP = 2
MAX_WIDTH = 640

if not DATASET_DIR.exists():
    raise FileNotFoundError(f'No se encontró {DATASET_DIR}. Sube el dataset a Drive y corrige DATASET_DIR.')

all_signs = sorted(p.name for p in DATASET_DIR.iterdir() if p.is_dir())
if N_SIGNS is None or N_SIGNS >= len(all_signs):
    SIGNS = all_signs
else:
    SIGNS = sorted(random.Random(SEED).sample(all_signs, N_SIGNS))

print(f'Señas disponibles: {len(all_signs)} | Señas usadas: {len(SIGNS)}')
print(SIGNS)

## 3. Extracción de características por video

Para cada video se detectan con MediaPipe Holistic las dos manos y los hombros. Con esos landmarks se construye una fila con:

* **Forma de la mano:** los 20 landmarks (sin la muñeca) en coordenadas locales (se resta la muñeca y se divide entre la distancia muñeca–nudillo medio), con su **media y desviación estándar** a lo largo del video. Reduce el efecto del tamaño y la posición de la mano en la imagen.
* **Movimiento de la muñeca:** posición respecto al centro de los hombros, normalizada con el ancho de los hombros. Se resume con posición inicial y final, desplazamiento, rangos, longitud del recorrido y velocidad media y máxima.
* **Detección:** proporción de frames donde se detectó cada mano. Distingue una mano ausente de una mano en una posición cualquiera.

Si una mano nunca se detecta, sus características quedan como `NaN` y no se rellenan con ceros en este paso.

In [ ]:
HOLISTIC_URL = ('https://storage.googleapis.com/mediapipe-models/holistic_landmarker/'
                'holistic_landmarker/float16/1/holistic_landmarker.task')
HOLISTIC_PATH = Path('/content/holistic_landmarker.task')
if not HOLISTIC_PATH.exists():
    urllib.request.urlretrieve(HOLISTIC_URL, HOLISTIC_PATH)

HAND_NAMES = ('left', 'right')


def lm_array(landmarks, n):
    out = np.full((n, 3), np.nan, dtype=np.float32)
    if landmarks:
        for i, lm in enumerate(landmarks[:n]):
            out[i] = [lm.x, lm.y, lm.z]
    return out


def extract_landmarks(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError('OpenCV no pudo abrir el video')
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not np.isfinite(fps) or fps <= 0:
        fps = 30.0

    options = vision.HolisticLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=str(HOLISTIC_PATH)),
        running_mode=vision.RunningMode.VIDEO,
        min_face_detection_confidence=0.5,
        min_face_landmarks_confidence=0.5,
        min_pose_detection_confidence=0.5,
        min_pose_landmarks_confidence=0.5,
        min_hand_landmarks_confidence=0.5,
        output_face_blendshapes=False,
        output_segmentation_mask=False,
    )

    hands, shoulders = [], []
    index = 0
    with vision.HolisticLandmarker.create_from_options(options) as model:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if index % SKIP == 0:
                if frame.shape[1] > MAX_WIDTH:
                    scale = MAX_WIDTH / frame.shape[1]
                    frame = cv2.resize(frame, None, fx=scale, fy=scale)
                rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                result = model.detect_for_video(image, int(index * 1000 / fps))
                hands.append(np.stack([lm_array(result.left_hand_landmarks, 21),
                                       lm_array(result.right_hand_landmarks, 21)]))
                shoulders.append(lm_array(result.pose_landmarks, 33)[[11, 12]])
            index += 1
    cap.release()

    if not hands:
        raise ValueError('El video no tiene frames legibles')
    return np.stack(hands), np.stack(shoulders), fps / SKIP


def hand_features(hand, center, width, fps, name):
    features = {}
    wrist = hand[:, 0, :]
    features[f'{name}_det_ratio'] = float(np.isfinite(wrist[:, 0]).mean())

    scale = np.linalg.norm(hand[:, 9, :] - wrist, axis=1)
    scale = np.where(scale > 1e-6, scale, np.nan)
    local = (hand[:, 1:, :] - wrist[:, None, :]) / scale[:, None, None]
    mean, std = np.nanmean(local, axis=0), np.nanstd(local, axis=0)
    for lm in range(20):
        for a, axis in enumerate('xyz'):
            features[f'{name}_p{lm + 1}_{axis}_mean'] = float(mean[lm, a])
            features[f'{name}_p{lm + 1}_{axis}_std'] = float(std[lm, a])

    body = (wrist[:, :2] - center[:, :2]) / width[:, None]
    valid_points = body[np.isfinite(body).all(axis=1)]
    names = ['start_x', 'start_y', 'end_x', 'end_y', 'delta_x', 'delta_y',
             'range_x', 'range_y', 'path_length', 'speed_mean', 'speed_max']
    values = dict.fromkeys(names, np.nan)
    if len(valid_points) >= 2:
        k = max(1, len(valid_points) // 5)
        start, end = valid_points[:k].mean(axis=0), valid_points[-k:].mean(axis=0)
        steps = np.linalg.norm(np.diff(body, axis=0), axis=1)
        steps = steps[np.isfinite(steps)]
        values.update({
            'start_x': start[0], 'start_y': start[1], 'end_x': end[0], 'end_y': end[1],
            'delta_x': end[0] - start[0], 'delta_y': end[1] - start[1],
            'range_x': np.ptp(valid_points[:, 0]), 'range_y': np.ptp(valid_points[:, 1]),
            'path_length': steps.sum() if steps.size else np.nan,
            'speed_mean': steps.mean() * fps if steps.size else np.nan,
            'speed_max': steps.max() * fps if steps.size else np.nan,
        })
    for key, value in values.items():
        features[f'{name}_wrist_{key}'] = float(value)
    return features


def video_features(hands, shoulders, fps):
    with warnings.catch_warnings(), np.errstate(all='ignore'):
        warnings.simplefilter('ignore')
        left_shoulder, right_shoulder = shoulders[:, 0], shoulders[:, 1]
        center = (left_shoulder + right_shoulder) / 2
        width = np.linalg.norm((left_shoulder - right_shoulder)[:, :2], axis=1)
        width = np.where(width > 1e-6, width, np.nan)
        features = {'n_frames': int(hands.shape[0]), 'duration_s': float(hands.shape[0] / fps)}
        for h, name in enumerate(HAND_NAMES):
            features.update(hand_features(hands[:, h], center, width, fps, name))
    return features

### Ejecución de la extracción

Es la parte lenta (depende del número de videos). Cada 25 videos se guarda el avance en Drive; si se interrumpe, basta con volver a ejecutar esta celda.

In [ ]:
EXTENSIONS = {'.mp4', '.mov'}

videos = []
for sign in SIGNS:
    files = sorted((p for p in (DATASET_DIR / sign).iterdir() if p.suffix.lower() in EXTENSIONS),
                   key=lambda p: p.name)
    if MAX_VIDEOS_PER_SIGN:
        files = files[:MAX_VIDEOS_PER_SIGN]
    videos.extend((sign, p) for p in files)
print('Videos a procesar:', len(videos))

rows = pd.read_csv(FEATURES_CSV).to_dict('records') if FEATURES_CSV.exists() else []
done = {r['video_id'] for r in rows}
failed = []

for sign, path in tqdm(videos):
    video_id = f'{sign}/{path.name}'
    if video_id in done:
        continue
    try:
        hands, shoulders, fps = extract_landmarks(path)
        rows.append({'video_id': video_id, 'label': sign, **video_features(hands, shoulders, fps)})
    except Exception as error:
        failed.append((video_id, str(error)))
        continue
    if len(rows) % 25 == 0:
        pd.DataFrame(rows).to_csv(FEATURES_CSV, index=False)

pd.DataFrame(rows).to_csv(FEATURES_CSV, index=False)
print(f'Videos con características: {len(rows)} | Con error: {len(failed)}')
for video_id, message in failed[:10]:
    print(' -', video_id, message)

## 4. Revisión rápida del dataset de características

Se verifica el tamaño, el balance de clases y la detección de manos. Los videos donde no se detectó ninguna mano no contienen información útil y se excluyen.

In [ ]:
df = pd.read_csv(FEATURES_CSV)
df = df[df['label'].isin(SIGNS)].reset_index(drop=True)
print('Forma:', df.shape)

no_hands = (df['left_det_ratio'] == 0) & (df['right_det_ratio'] == 0)
print('Videos sin ninguna mano detectada (se excluyen):', int(no_hands.sum()))
df = df[~no_hands].reset_index(drop=True)

counts = df['label'].value_counts()
print(f'Videos: {len(df)} | Clases: {df["label"].nunique()} | '
      f'Mín/Máx por clase: {counts.min()}/{counts.max()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
counts.plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Videos por seña')
axes[0].set_ylabel('Videos')
df[['left_det_ratio', 'right_det_ratio']].plot.hist(bins=20, alpha=0.6, ax=axes[1])
axes[1].set_title('Proporción de frames con mano detectada')
plt.tight_layout()
plt.show()

## 5. Partición de datos y validación

* Se reserva **20 % para prueba** (estratificado por seña). Solo se usa al final, una vez.
* Todos los modelos se comparan con **los mismos 5 folds** estratificados sobre el 80 % de entrenamiento.
* El `Pipeline` aplica: imputación (ausente = 0) → filtro de varianza → estandarización → modelo. Todo se ajusta solo con el fold de entrenamiento.
* **¿Por qué estandarizar?** SVM, K vecinos, regresión logística y MLP dependen de la escala de las variables; con ella, ninguna característica domina solo por su rango.

In [ ]:
META = ['video_id', 'label']
X = df.drop(columns=META)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print(f'Entrenamiento: {X_train.shape} | Prueba: {X_test.shape}')
print(f'Características: {X.shape[1]}')


def make_pipe(clf):
    return Pipeline([
        ('imp', SimpleImputer(strategy='constant', fill_value=0.0, keep_empty_features=True)),
        ('var', VarianceThreshold(threshold=1e-8)),
        ('scale', StandardScaler()),
        ('clf', clf),
    ])

## 6. Comparativa de 6 modelos individuales

Se eligieron algoritmos de familias distintas, todos individuales (sin ensambles):

| Modelo | Por qué se incluye |
|---|---|
| Regresión logística | Referencia lineal, rápida y fácil de interpretar |
| SVM (kernel RBF) | Suele funcionar bien con pocos datos y muchas características numéricas |
| K vecinos más cercanos | Clasifica por similitud con otros videos; sin entrenamiento explícito |
| Árbol de decisión | Reglas interpretables, no necesita escalado |
| Naive Bayes gaussiano | Referencia probabilística muy rápida |
| MLP | Red neuronal que captura relaciones no lineales |

Se añade un **clasificador de referencia** (`DummyClassifier`) que predice la clase más frecuente, para saber qué rendimiento se obtiene sin aprender nada. No cuenta entre los 6 modelos.

In [ ]:
MODELOS = {
    'Regresión logística': LogisticRegression(max_iter=2000, random_state=SEED),
    'SVM': SVC(kernel='rbf', random_state=SEED),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Árbol de decisión': DecisionTreeClassifier(random_state=SEED),
    'Naive Bayes': GaussianNB(),
    'MLP': MLPClassifier(hidden_layer_sizes=(128,), max_iter=1000, random_state=SEED),
}

SCORING = {'f1_macro': 'f1_macro', 'accuracy': 'accuracy', 'recall_macro': 'recall_macro'}


def evaluate_cv(name, clf):
    result = cross_validate(make_pipe(clf), X_train, y_train, cv=cv, scoring=SCORING,
                            n_jobs=-1, return_train_score=True)
    return {
        'modelo': name,
        'f1_macro': result['test_f1_macro'].mean(),
        'f1_macro_std': result['test_f1_macro'].std(),
        'accuracy': result['test_accuracy'].mean(),
        'recall_macro': result['test_recall_macro'].mean(),
        'f1_macro_train': result['train_f1_macro'].mean(),
        'tiempo_entrenamiento_s': result['fit_time'].mean(),
    }


baseline = evaluate_cv('Referencia (clase más frecuente)', DummyClassifier(strategy='most_frequent'))
comparativa = pd.DataFrame([evaluate_cv(n, m) for n, m in MODELOS.items()])
comparativa = comparativa.sort_values('f1_macro', ascending=False).set_index('modelo')

print('Validación cruzada de 5 folds sobre entrenamiento (ordenado por F1 macro):')
display(comparativa.round(4))
print(f"Referencia sin aprender: F1 macro = {baseline['f1_macro']:.4f}, accuracy = {baseline['accuracy']:.4f}")

plt.figure(figsize=(9, 4))
plt.barh(comparativa.index[::-1], comparativa['f1_macro'][::-1],
         xerr=comparativa['f1_macro_std'][::-1], color='steelblue')
plt.axvline(baseline['f1_macro'], color='red', linestyle='--', label='Referencia')
plt.xlabel('F1 macro (media ± desviación en 5 folds)')
plt.title('Comparativa de modelos')
plt.legend()
plt.tight_layout()
plt.show()

### Lectura de la comparativa

La columna `f1_macro_train` frente a `f1_macro` indica sobreajuste: si el modelo acierta casi todo en entrenamiento y mucho menos en validación, está memorizando. La desviación entre folds muestra qué tan estable es cada modelo.

*(Completar con los resultados observados al ejecutar: qué modelos quedaron arriba, cuáles se sobreajustan y cuál es el más rápido.)*

## 7. Ajuste fino de los dos mejores

Se toman automáticamente los **dos modelos con mayor F1 macro** de la comparativa y se buscan sus hiperparámetros con `GridSearchCV` (mismos folds, métrica F1 macro). Cada malla cubre los hiperparámetros que más influyen en el modelo:

* **Regresión logística / SVM:** `C` (regularización), `class_weight`, `kernel` y `gamma`.
* **KNN:** número de vecinos, ponderación por distancia y métrica.
* **Árbol:** profundidad máxima, mínimo de muestras por hoja y criterio.
* **Naive Bayes:** suavizado de varianza.
* **MLP:** arquitectura, regularización `alpha` y tasa de aprendizaje.

In [ ]:
GRIDS = {
    'Regresión logística': {
        'clf__C': [0.01, 0.1, 1, 10, 100],
        'clf__class_weight': [None, 'balanced'],
    },
    'SVM': {
        'clf__C': [0.1, 1, 10, 100],
        'clf__kernel': ['rbf', 'linear'],
        'clf__gamma': ['scale', 0.001, 0.01],
    },
    'KNN': {
        'clf__n_neighbors': [1, 3, 5, 7, 11, 15],
        'clf__weights': ['uniform', 'distance'],
        'clf__metric': ['euclidean', 'manhattan'],
    },
    'Árbol de decisión': {
        'clf__max_depth': [None, 5, 10, 20],
        'clf__min_samples_leaf': [1, 2, 5],
        'clf__criterion': ['gini', 'entropy'],
    },
    'Naive Bayes': {
        'clf__var_smoothing': list(np.logspace(-11, -5, 7)),
    },
    'MLP': {
        'clf__hidden_layer_sizes': [(64,), (128,), (128, 64)],
        'clf__alpha': [1e-4, 1e-3, 1e-2],
        'clf__learning_rate_init': [1e-3, 1e-2],
    },
}

top2 = list(comparativa.index[:2])
print('Modelos a ajustar:', top2)

tuned, tuning_rows = {}, []
for name in top2:
    search = GridSearchCV(make_pipe(MODELOS[name]), GRIDS[name], scoring='f1_macro',
                          cv=cv, n_jobs=-1, refit=True)
    start = time.perf_counter()
    search.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    tuned[name] = search
    tuning_rows.append({
        'modelo': name,
        'f1_macro_base': comparativa.loc[name, 'f1_macro'],
        'f1_macro_ajustado': search.best_score_,
        'mejora': search.best_score_ - comparativa.loc[name, 'f1_macro'],
        'combinaciones': len(search.cv_results_['params']),
        'tiempo_busqueda_s': elapsed,
        'mejores_hiperparametros': {k.replace('clf__', ''): v for k, v in search.best_params_.items()},
    })

tuning = pd.DataFrame(tuning_rows).set_index('modelo')
pd.set_option('display.max_colwidth', None)
display(tuning.round(4))

## 8. Elección del modelo individual final

Los dos modelos ajustados se evalúan en el **conjunto de prueba reservado**, que no se usó para elegir ni ajustar nada. La decisión se toma con la validación cruzada, no con la prueba, para no sesgar la elección.

**Criterio:** se elige el modelo con mayor F1 macro en validación cruzada. Si los dos están a menos de **1 punto** de diferencia, se prefiere el de menor tiempo de entrenamiento, ya que el rendimiento es equivalente y es más barato de mantener. También se reportan el tiempo de predicción por video y la interpretabilidad.

In [ ]:
INTERPRETABILIDAD = {
    'Regresión logística': 'Alta (coeficientes por clase)',
    'SVM': 'Baja (frontera no lineal)',
    'KNN': 'Media (basado en ejemplos similares)',
    'Árbol de decisión': 'Alta (reglas explícitas)',
    'Naive Bayes': 'Media (probabilidades por característica)',
    'MLP': 'Baja (caja negra)',
}

final_rows = []
for name, search in tuned.items():
    model = search.best_estimator_
    start = time.perf_counter()
    pred = model.predict(X_test)
    predict_ms = (time.perf_counter() - start) / len(X_test) * 1000
    final_rows.append({
        'modelo': name,
        'f1_macro_cv': search.best_score_,
        'f1_macro_train': f1_score(y_train, model.predict(X_train), average='macro'),
        'f1_macro_prueba': f1_score(y_test, pred, average='macro'),
        'accuracy_prueba': accuracy_score(y_test, pred),
        'recall_macro_prueba': recall_score(y_test, pred, average='macro'),
        'tiempo_entrenamiento_s': search.refit_time_,
        'prediccion_ms_por_video': predict_ms,
        'interpretabilidad': INTERPRETABILIDAD[name],
    })

final_table = pd.DataFrame(final_rows).set_index('modelo')
display(final_table.round(4))

best_cv = final_table['f1_macro_cv'].max()
candidates = final_table[final_table['f1_macro_cv'] >= best_cv - 0.01]
FINAL_NAME = candidates['tiempo_entrenamiento_s'].idxmin()
final_model = tuned[FINAL_NAME].best_estimator_
print(f'Modelo individual final: {FINAL_NAME}')

In [ ]:
final_pred = final_model.predict(X_test)
print(f'Modelo final: {FINAL_NAME}')
print(classification_report(y_test, final_pred, zero_division=0))

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_test, final_pred, ax=ax, cmap='Blues',
                                        xticks_rotation=90, colorbar=False)
ax.set_title(f'Matriz de confusión en prueba - {FINAL_NAME}')
plt.tight_layout()
plt.show()

joblib.dump(final_model, WORK_DIR / 'modelo_final_avance4.joblib')
with open(WORK_DIR / 'resumen_avance4.json', 'w', encoding='utf-8') as file:
    json.dump({
        'modelo_final': FINAL_NAME,
        'senas': SIGNS,
        'videos': int(len(df)),
        'f1_macro_prueba': float(f1_score(y_test, final_pred, average='macro')),
        'accuracy_prueba': float(accuracy_score(y_test, final_pred)),
    }, file, ensure_ascii=False, indent=2)

### Argumentación de la elección

*(Completar con los resultados observados al ejecutar. Incluir: por qué se eligió este modelo además de su F1 macro, cómo se compara con el otro finalista en tiempo de entrenamiento, tiempo de predicción e interpretabilidad, qué señas se confunden entre sí en la matriz de confusión y si la distancia entre F1 de entrenamiento, validación y prueba indica sobreajuste.)*

## 9. Conclusiones y siguientes pasos

*(Completar tras ejecutar.)* Puntos a cubrir:

* Rendimiento del modelo final frente a la referencia (`DummyClassifier`) y frente al baseline del Avance 3, recordando que aquel solo clasificaba la letra A.
* Qué mejora aportó el ajuste de hiperparámetros y si el espacio de búsqueda fue suficiente.
* Limitaciones: separación por video y no por participante, subconjunto de señas utilizado, videos sin manos detectadas.
* Siguiente etapa: ampliar a todas las señas, evaluar por participante si hay metadatos, y probar modelos de secuencias o ensambles.

## Declaración sobre el uso de herramientas de inteligencia artificial

*(Obligatoria. Sustituir este bloque por la plantilla de declaración en Markdown que adjunta la actividad en Experiencia 21, completándola con la información del equipo.)*

Información mínima a declarar:

* Herramientas utilizadas (por ejemplo, GitHub Copilot en VS Code) y su versión o modelo.
* Para qué se usaron: exploración del repositorio, diseño de la estructura de la libreta y generación de código base.
* Qué validó el equipo: revisión del código, ejecución secuencial completa y verificación de los resultados.
* Qué decisiones son responsabilidad del equipo: elección de modelos, métrica y justificación de conclusiones.